In [1]:
import pandas as pd
import numpy as np
import random
from datetime import datetime, timedelta

# ==========================================
# CONFIGURATION
# ==========================================
NUM_ROWS = 20000
YEAR = 2025

# Centers
CENTERS = ['C001', 'C002', 'C003', 'C004', 'C005']
CENTER_NAMES = {
    'C001': 'Delhi Apex Lab',
    'C002': 'Mumbai Andheri Zone',
    'C003': 'Kolkata Logistics Hub', # The Problem Child
    'C004': 'Bengaluru Tech Unit',
    'C005': 'Chandigarh Satellite'   # The Star Performer
}

# Test Types & Expected TAT (Minutes)
TEST_TYPES = {
    'T01': {'name': 'CBC (Blood)', 'tat': 60, 'complexity': 'low'},
    'T02': {'name': 'X-Ray Chest', 'tat': 120, 'complexity': 'medium'},
    'T03': {'name': 'MRI Brain', 'tat': 240, 'complexity': 'high'},
    'T04': {'name': 'Lipid Profile', 'tat': 90, 'complexity': 'low'},
    'T05': {'name': 'COVID RT-PCR', 'tat': 360, 'complexity': 'variable'}
}

# Statuses
STATUS_OPTS = ['S05', 'S06', 'S03'] # Completed, Cancelled, Processing
STATUS_WEIGHTS = [0.94, 0.04, 0.02] # Mostly completed for good analysis

# Patient IDs (Sample pool)
PATIENT_IDS = [f'P{str(i).zfill(5)}' for i in range(1, 3501)]

# ==========================================
# HELPER FUNCTIONS
# ==========================================

def get_seasonal_weight(month):
    """
    Creates a 'Monsoon Surge' story.
    High volume & stress in July(7), Aug(8), Sept(9).
    Low volume in Dec(12), Jan(1).
    """
    if month in [7, 8, 9]: return 1.6  # Peak Season
    if month in [5, 6]: return 1.2     # Pre-season
    if month in [1, 12]: return 0.8    # Low season
    return 1.0

def generate_random_date(year):
    """Generates a date with weighted probability for seasonality"""
    start_date = datetime(year, 1, 1)
    # Generate a random day, but reject/accept based on month weight
    while True:
        day_offset = random.randint(0, 364)
        curr_date = start_date + timedelta(days=day_offset)
        weight = get_seasonal_weight(curr_date.month)
        # Simple rejection sampling
        if random.random() < (weight / 1.6): 
            return curr_date

def calculate_stage_duration(base_minutes, complexity, center_id, is_mismatch, is_peak_season):
    """
    Calculates realistic duration with penalties for:
    1. Problem Centers (Kolkata)
    2. Peak Season (July-Sept)
    3. Logistics Mismatches
    """
    # Base noise
    duration = np.random.normal(base_minutes * 0.8, base_minutes * 0.1)
    
    # 1. Center Factor
    if center_id == 'C003': # Kolkata (Problem)
        duration += random.uniform(10, 40)
    elif center_id == 'C005': # Chandigarh (Efficient)
        duration -= random.uniform(5, 15)
        
    # 2. Peak Season Stress Factor (Overloaded Labs)
    if is_peak_season:
        duration += random.uniform(0, base_minutes * 0.4)
        
    # 3. Mismatch Penalty (Only affects Stage 1 usually, but adding general friction)
    if is_mismatch:
        duration += random.uniform(20, 60)
        
    return max(5, int(duration)) # Minimum 5 mins

# ==========================================
# DATA GENERATION
# ==========================================

data = []

print(f"Generating {NUM_ROWS} rows of realistic data...")

for i in range(1, NUM_ROWS + 1):
    test_id = f"TEST-{str(i).zfill(7)}"
    
    # 1. Date & Seasonality
    test_date = generate_random_date(YEAR)
    is_peak_season = test_date.month in [7, 8, 9]
    
    # 2. Location Logic (Problem Center vs Others)
    # Weighted choice for centers to ensure Kolkata (C003) has enough volume to show the problem
    proc_center = np.random.choice(CENTERS, p=[0.2, 0.2, 0.25, 0.2, 0.15]) 
    
    # Mismatch Logic: 15% chance of mismatch
    # If mismatch, Actual Center is random other center
    if random.random() < 0.15:
        actual_center = random.choice([c for c in CENTERS if c != proc_center])
        match_flag = "Mismatch"
    else:
        actual_center = proc_center
        match_flag = "Match"
        
    # 3. Test Type
    t_id = random.choice(list(TEST_TYPES.keys()))
    t_info = TEST_TYPES[t_id]
    
    # 4. Status
    status = np.random.choice(STATUS_OPTS, p=STATUS_WEIGHTS)
    
    # 5. Timing & Stages
    # Stage 1: Logistics (Affected heavily by Mismatch)
    start_time = datetime.combine(test_date, datetime.min.time()) + timedelta(minutes=random.randint(480, 1000)) # 8 AM to 4 PM
    
    stage1_base = 30 # Standard logistics time
    if match_flag == "Mismatch": stage1_base += 90 # Huge penalty for mismatch
    
    s1_dur = calculate_stage_duration(stage1_base, 'low', proc_center, match_flag == "Mismatch", is_peak_season)
    s1_end = start_time + timedelta(minutes=s1_dur)
    
    # Stage 2: Processing (The main work, affected by Test Complexity & Center Efficiency)
    s2_start = s1_end + timedelta(minutes=random.randint(5, 20)) # Gap
    s2_dur = calculate_stage_duration(t_info['tat'] * 0.7, t_info['complexity'], proc_center, False, is_peak_season)
    
    # Make Kolkata (C003) specifically bad at MRI (T03) processing
    if proc_center == 'C003' and t_id == 'T03':
        s2_dur += random.randint(30, 90)
        
    s2_end = s2_start + timedelta(minutes=s2_dur)
    
    # Stage 3: Reporting (Usually fast)
    s3_start = s2_end + timedelta(minutes=random.randint(5, 15))
    s3_dur = int(np.random.normal(15, 5))
    if s3_dur < 5: s3_dur = 5
    s3_end = s3_start + timedelta(minutes=s3_dur)
    
    # 6. TAT Calculation
    total_tat = (s3_end - start_time).total_seconds() / 60
    expected_tat = t_info['tat']
    
    # Status Checks
    if status == 'S06': # Cancelled
        total_tat = 0 # Or NULL logic
        tat_missing = True
        out_tat_flag = False
        under_tat_flag = False
        # Nullify end dates for realism if needed, but keeping filled for simplicity in some BI tools
    elif status == 'S03': # Processing
        total_tat = 0
        tat_missing = True
        out_tat_flag = False
        under_tat_flag = False
    else: # Completed
        tat_missing = False
        if total_tat > expected_tat:
            out_tat_flag = True
            under_tat_flag = False
        else:
            out_tat_flag = False
            under_tat_flag = True

    # 7. Append Row
    row = [
        test_id,
        test_date.strftime('%Y-%m-%d'),
        random.choice(PATIENT_IDS),
        proc_center,
        actual_center,
        t_id,
        status,
        start_time.strftime('%Y-%m-%d %H:%M'),
        s1_end.strftime('%Y-%m-%d %H:%M'),
        s2_start.strftime('%Y-%m-%d %H:%M'),
        s2_end.strftime('%Y-%m-%d %H:%M'),
        s3_start.strftime('%Y-%m-%d %H:%M'),
        s3_end.strftime('%Y-%m-%d %H:%M'),
        int(total_tat),
        expected_tat,
        under_tat_flag,
        out_tat_flag,
        tat_missing,
        match_flag
    ]
    data.append(row)

# ==========================================
# EXPORT
# ==========================================

columns = [
    "Test_ID", "Test_Date", "Patient_ID", "Processing_Center_ID", 
    "Actual_Test_Center_ID", "TestType_ID", "Status_ID", 
    "Stage1_Start", "Stage1_End", "Stage2_Start", "Stage2_End", 
    "Stage3_Start", "Stage3_End", "Total_TAT_Minutes", 
    "Expected_TAT_Minutes", "Under_TAT_Flag", "Out_TAT_Flag", 
    "TAT_Missing_Flag", "Center_Match_Flag"
]

df = pd.DataFrame(data, columns=columns)

# Final Validity Check
breach_rate = df[df['Out_TAT_Flag'] == True].shape[0] / df.shape[0]
print(f"Dataset Generated successfully.")
print(f"Total Rows: {len(df)}")
print(f"Approx Breach Rate: {breach_rate:.2%}") # Should be around 14-16% based on logic

# Save
filename = "Fact_Test_Performance.csv"
df.to_csv(filename, index=False)
print(f"File saved as: {filename}")

Generating 20000 rows of realistic data...
Dataset Generated successfully.
Total Rows: 20000
Approx Breach Rate: 63.68%
File saved as: Fact_Test_Performance.csv
